# Cat Overlays and Gaussian Shading Watermark Robustness

Self-contained notebook: place visually subtle overlays (especially cats) on Gaussian Shading
watermarked images, measure how much the **VAE latent** moves, and on a smaller subset run
**actual DDIM inversion + sign-bit watermark decoding**.

This repo does **not** ship the full cryptographic Gaussian Shading stack (ChaCha20 keystream,
diffusion of message bits, etc.). Watermark bits here follow the same **sign rule** used elsewhere
in the project:

$$b = \mathbf{1}[z \ge 0],\qquad \hat b = \mathbf{1}[\hat z \ge 0],\qquad
\text{bit\_accuracy} = \mathrm{mean}(\hat b = b).$$

If you later point `EXTERNAL_GS_DECODE_FN` at a real decoder, the watermark section will use it.


## 1. Configuration


In [ ]:
from __future__ import annotations

import gc
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageFont

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# ---- paths (edit these) ----
OUTPUT_DIR = Path("results/cat_overlay_gs")
WATERMARKED_DIR = OUTPUT_DIR / "watermarked"
EXTERNAL_IMAGE_DIR = Path("data/watermarked_images")
EXTERNAL_BITS_PATH = Path("data/watermark_bits.pt")
EXTERNAL_KEY_PATH = Path("data/watermark_key.json")
CAT_PNG_PATH = Path("assets/cat_overlay.png")

# ---- experiment size ----
SEED = 42
IMAGE_SIZE = 512
NUM_IMAGES = 50
NUM_WATERMARK_IMAGES = 8
MODEL_ID = "runwayml/stable-diffusion-v1-5"
PROMPT = "a photograph of an astronaut riding a horse"
GUIDANCE_SCALE = 7.5
NUM_INFERENCE_STEPS = 50
INVERSION_STRENGTH = 1.0

CAT_SIZES = [0.02, 0.05, 0.10, 0.20, 0.40, 0.60]
CAT_OPACITIES = [0.01, 0.02, 0.05, 0.10, 0.20, 0.40, 0.80, 1.00]
CAT_POSITIONS = ["center", "random"]

NOISE_STDS = [0.01, 0.05, 0.10]
SHAPE_OPACITIES = [0.05, 0.20, 0.80]
TEXT_OPACITIES = [0.20, 0.80]

WM_CAT_SIZES = [0.05, 0.20, 0.60]
WM_CAT_OPACITIES = [0.05, 0.20, 0.80]
WM_CAT_POSITIONS = ["center"]
WM_INCLUDE_CONTROLS = True

# Set True for a tiny local smoke test
QUICK_MODE = False
if QUICK_MODE:
    NUM_IMAGES = 4
    NUM_WATERMARK_IMAGES = 2
    CAT_SIZES = [0.05, 0.20]
    CAT_OPACITIES = [0.05, 0.40]
    CAT_POSITIONS = ["center"]
    WM_CAT_SIZES = [0.20]
    WM_CAT_OPACITIES = [0.40]
    NOISE_STDS = [0.05]
    SHAPE_OPACITIES = [0.20]
    TEXT_OPACITIES = [0.40]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "examples").mkdir(parents=True, exist_ok=True)
WATERMARKED_DIR.mkdir(parents=True, exist_ok=True)
CAT_PNG_PATH.parent.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

print(f"device={DEVICE}, dtype={DTYPE}, NUM_IMAGES={NUM_IMAGES}, QUICK_MODE={QUICK_MODE}")
print(f"OUTPUT_DIR={OUTPUT_DIR.resolve()}")
print(f"CAT_PNG_PATH={CAT_PNG_PATH.resolve()}")


## 2. Helpers: cat asset, overlays, metrics

Creates a simple transparent cat silhouette if `CAT_PNG_PATH` is missing, then defines alpha
compositing and latent/image metrics.


In [ ]:
def ensure_cat_png(path: Path, size: int = 512) -> Path:
    """Write a simple sitting-cat silhouette with alpha if the file is absent."""
    if path.is_file():
        return path
    img = Image.new("RGBA", (size, size), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)
    draw.ellipse([110, 220, 400, 460], fill=(20, 20, 20, 255))
    draw.ellipse([250, 80, 420, 260], fill=(20, 20, 20, 255))
    draw.polygon([(270, 120), (290, 30), (330, 100)], fill=(20, 20, 20, 255))
    draw.polygon([(360, 100), (400, 30), (410, 120)], fill=(20, 20, 20, 255))
    draw.polygon([(110, 320), (30, 260), (70, 340), (120, 360)], fill=(20, 20, 20, 255))
    draw.ellipse([300, 140, 325, 165], fill=(240, 240, 240, 220))
    draw.ellipse([350, 140, 375, 165], fill=(240, 240, 240, 220))
    img.save(path)
    print(f"wrote placeholder cat silhouette -> {path}")
    return path


CAT_PNG_PATH = ensure_cat_png(CAT_PNG_PATH)
cat_rgba = Image.open(CAT_PNG_PATH).convert("RGBA")
print(f"cat overlay loaded: size={cat_rgba.size}, mode={cat_rgba.mode}")


def pil_to_float(img: Image.Image) -> np.ndarray:
    return np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0


def float_to_pil(arr: np.ndarray) -> Image.Image:
    clipped = np.clip(arr, 0.0, 1.0)
    return Image.fromarray((clipped * 255.0).round().astype(np.uint8), mode="RGB")


def alpha_composite_rgba(
    base_rgb: Image.Image,
    overlay_rgba: Image.Image,
    xy: tuple[int, int],
    opacity: float,
) -> Image.Image:
    base = base_rgb.convert("RGBA")
    ov = overlay_rgba.copy()
    if opacity < 1.0:
        alpha = np.asarray(ov.split()[-1], dtype=np.float32)
        alpha = np.clip(alpha * float(opacity), 0, 255).astype(np.uint8)
        ov.putalpha(Image.fromarray(alpha, mode="L"))
    layer = Image.new("RGBA", base.size, (0, 0, 0, 0))
    layer.paste(ov, xy, ov)
    return Image.alpha_composite(base, layer).convert("RGB")


def place_cat(
    base: Image.Image,
    cat: Image.Image,
    size_frac: float,
    opacity: float,
    position: str,
    rng: np.random.Generator,
) -> Image.Image:
    W, H = base.size
    target_w = max(1, int(round(W * size_frac)))
    cw, ch = cat.size
    target_h = max(1, int(round(target_w * (ch / max(cw, 1)))))
    if target_h > H:
        scale = H / target_h
        target_w = max(1, int(round(target_w * scale)))
        target_h = max(1, int(round(target_h * scale)))
    resized = cat.resize((target_w, target_h), Image.Resampling.LANCZOS)
    if position == "center":
        x = (W - target_w) // 2
        y = (H - target_h) // 2
    elif position == "random":
        x = int(rng.integers(0, max(1, W - target_w + 1)))
        y = int(rng.integers(0, max(1, H - target_h + 1)))
    else:
        raise ValueError(f"unknown position {position}")
    return alpha_composite_rgba(base, resized, (x, y), opacity)


def overlay_gaussian_noise(base: Image.Image, std: float, rng: np.random.Generator) -> Image.Image:
    arr = pil_to_float(base)
    noise = rng.normal(0.0, std, size=arr.shape).astype(np.float32)
    return float_to_pil(arr + noise)


def overlay_random_shapes(
    base: Image.Image,
    opacity: float,
    rng: np.random.Generator,
    n_shapes: int = 5,
) -> Image.Image:
    W, H = base.size
    layer = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    draw = ImageDraw.Draw(layer)
    for _ in range(n_shapes):
        color = tuple(int(c) for c in rng.integers(0, 256, size=3)) + (int(round(255 * opacity)),)
        shape = str(rng.choice(["ellipse", "rectangle", "polygon"]))
        x0 = int(rng.integers(0, W))
        y0 = int(rng.integers(0, H))
        x1 = int(np.clip(x0 + rng.integers(20, max(21, W // 3)), 0, W - 1))
        y1 = int(np.clip(y0 + rng.integers(20, max(21, H // 3)), 0, H - 1))
        if shape == "ellipse":
            draw.ellipse([x0, y0, x1, y1], fill=color)
        elif shape == "rectangle":
            draw.rectangle([x0, y0, x1, y1], fill=color)
        else:
            pts = [
                (int(rng.integers(min(x0, x1), max(x0, x1) + 1)),
                 int(rng.integers(min(y0, y1), max(y0, y1) + 1)))
                for _ in range(3)
            ]
            draw.polygon(pts, fill=color)
    return Image.alpha_composite(base.convert("RGBA"), layer).convert("RGB")


def overlay_text(
    base: Image.Image,
    opacity: float,
    rng: np.random.Generator,
    text: str = "WM",
) -> Image.Image:
    W, H = base.size
    layer = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    draw = ImageDraw.Draw(layer)
    try:
        font = ImageFont.truetype("arial.ttf", size=max(16, W // 10))
    except OSError:
        font = ImageFont.load_default()
    bbox = draw.textbbox((0, 0), text, font=font)
    tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
    x = int(rng.integers(0, max(1, W - tw)))
    y = int(rng.integers(0, max(1, H - th)))
    fill = (255, 255, 255, int(round(255 * opacity)))
    draw.text((x, y), text, fill=fill, font=font)
    return Image.alpha_composite(base.convert("RGBA"), layer).convert("RGB")


def image_to_vae_tensor(img: Image.Image, device: str, dtype: torch.dtype) -> torch.Tensor:
    arr = pil_to_float(img)
    t = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    t = t * 2.0 - 1.0
    return t.to(device=device, dtype=dtype)


@torch.no_grad()
def encode_vae_mean(vae, img: Image.Image, device: str, dtype: torch.dtype) -> torch.Tensor:
    x = image_to_vae_tensor(img, device, dtype)
    posterior = vae.encode(x).latent_dist
    z = posterior.mean * vae.config.scaling_factor
    return z.detach().float().cpu()


@torch.no_grad()
def decode_vae(vae, z: torch.Tensor, device: str, dtype: torch.dtype) -> Image.Image:
    lat = z.to(device=device, dtype=dtype) / vae.config.scaling_factor
    dec = vae.decode(lat).sample
    dec = ((dec + 1.0) * 0.5).clamp(0, 1)
    arr = dec[0].detach().float().cpu().permute(1, 2, 0).numpy()
    return float_to_pil(arr)


def latent_metrics(z0: torch.Tensor, z1: torch.Tensor) -> dict:
    a = z0.reshape(-1).numpy().astype(np.float64)
    b = z1.reshape(-1).numpy().astype(np.float64)
    diff = b - a
    l2 = float(np.linalg.norm(diff))
    rel = l2 / max(float(np.linalg.norm(a)), 1e-12)
    cos = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))
    rmse = float(np.sqrt(np.mean(diff ** 2)))
    return {
        "latent_l2": l2,
        "latent_l2_rel": rel,
        "latent_cosine_distance": 1.0 - cos,
        "latent_rmse": rmse,
    }


def latent_heatmap(z0: torch.Tensor, z1: torch.Tensor) -> np.ndarray:
    d = (z1 - z0).squeeze(0).numpy()
    return np.sqrt((d ** 2).sum(axis=0))


def image_metrics(img0: Image.Image, img1: Image.Image, lpips_fn=None, device="cpu") -> dict:
    a = pil_to_float(img0)
    b = pil_to_float(img1)
    mse = float(np.mean((a - b) ** 2))
    psnr = float(10.0 * np.log10(1.0 / max(mse, 1e-12)))
    out = {"pixel_mse": mse, "psnr": psnr, "ssim": float("nan"), "lpips": float("nan")}
    try:
        from skimage.metrics import structural_similarity
        out["ssim"] = float(structural_similarity(a, b, channel_axis=2, data_range=1.0))
    except Exception:
        pass
    if lpips_fn is not None:
        t0 = torch.from_numpy(a).permute(2, 0, 1).unsqueeze(0) * 2 - 1
        t1 = torch.from_numpy(b).permute(2, 0, 1).unsqueeze(0) * 2 - 1
        with torch.no_grad():
            out["lpips"] = float(lpips_fn(t0.to(device), t1.to(device)).item())
    return out


def bits_from_latent(z: torch.Tensor | np.ndarray) -> np.ndarray:
    arr = z.detach().cpu().numpy() if torch.is_tensor(z) else np.asarray(z)
    return (arr >= 0).astype(np.uint8)


def bit_accuracy(recovered: np.ndarray, known: np.ndarray) -> float:
    return float(np.mean(recovered.reshape(-1) == known.reshape(-1)))


def condition_salt(condition: str) -> int:
    return int(np.frombuffer(condition.encode("utf-8"), dtype=np.uint8).sum())


lpips_fn = None
try:
    import lpips
    lpips_fn = lpips.LPIPS(net="alex")
    if DEVICE == "cuda":
        lpips_fn = lpips_fn.to(DEVICE)
    lpips_fn.eval()
    print("LPIPS available (alex).")
except Exception as exc:
    print(f"LPIPS unavailable ({type(exc).__name__}: {exc}); lpips column will be NaN.")

# Optional hook: decode_fn(image: PIL.Image, key: dict) -> np.ndarray bits
EXTERNAL_GS_DECODE_FN = None


## 3. Load or generate watermarked images

Preference order:

1. External directory (`EXTERNAL_IMAGE_DIR` + bits/key) if present.
2. Cached corpus under `WATERMARKED_DIR` (regenerates if it is only a synthetic placeholder and CUDA is available).
3. Generate with SD1.5 from known Gaussian latents. Known bits are the latent signs; the key records seeds + scheme.

Without CUDA and without external data, a synthetic placeholder corpus is created so the VAE overlay
section can still be exercised. That placeholder is **not** Gaussian Shading.


In [ ]:
def _save_corpus(images, bits_list, key, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    bits_map = {}
    for i, (img, bits) in enumerate(zip(images, bits_list)):
        stem = f"sample_{i:03d}"
        img.save(out_dir / f"{stem}.png")
        bits_map[stem] = torch.from_numpy(np.asarray(bits).astype(np.uint8))
    torch.save(bits_map, out_dir / "watermark_bits.pt")
    (out_dir / "watermark_key.json").write_text(json.dumps(key, indent=2), encoding="utf-8")


def _load_corpus_from_dir(image_dir: Path, bits_path: Path, key_path: Path):
    bits_obj = torch.load(bits_path, map_location="cpu", weights_only=False)
    key = json.loads(key_path.read_text(encoding="utf-8")) if key_path.is_file() else {}
    stems = sorted(bits_obj.keys())
    images, bits_list, names = [], [], []
    for stem in stems:
        png = image_dir / f"{stem}.png"
        if not png.is_file():
            pngs = [
                p for p in image_dir.glob(f"{stem}*")
                if p.suffix.lower() in {".png", ".jpg", ".jpeg"}
            ]
            if not pngs:
                continue
            png = pngs[0]
        img = Image.open(png).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
        images.append(img)
        bits_list.append(np.asarray(bits_obj[stem]).astype(np.uint8))
        names.append(stem)
    return images, bits_list, names, key


def generate_synthetic_placeholder_corpus(n: int):
    """CPU fallback for wiring checks only. Not real GS-watermarked SD images."""
    rng = np.random.default_rng(SEED)
    images, bits_list, names = [], [], []
    for i in range(n):
        arr = rng.integers(0, 256, size=(IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
        images.append(Image.fromarray(arr, mode="RGB"))
        bits = (rng.random((1, 4, IMAGE_SIZE // 8, IMAGE_SIZE // 8)) < 0.5).astype(np.uint8)
        bits_list.append(bits)
        names.append(f"sample_{i:03d}")
    key = {
        "scheme": "synthetic_placeholder",
        "note": "Random RGB + random bits for offline wiring only. Not Gaussian Shading.",
        "seeds": list(range(n)),
    }
    _save_corpus(images, bits_list, key, WATERMARKED_DIR)
    print(f"wrote SYNTHETIC placeholder corpus n={n} -> {WATERMARKED_DIR}")
    return images, bits_list, names, key


def generate_watermarked_corpus(n: int, seeds: list[int]):
    if DEVICE != "cuda":
        print(
            "CUDA unavailable: cannot generate real SD watermarked images. "
            "Using synthetic placeholder corpus so the VAE overlay section can still run. "
            "Provide EXTERNAL_* paths or run on GPU for real GS."
        )
        return generate_synthetic_placeholder_corpus(n)

    from experiments.generate_and_invert import (
        _load_generation_pipeline,
        _release_pipeline,
        _sample_latent,
    )

    pipe = _load_generation_pipeline(MODEL_ID, DTYPE, DEVICE)
    images, bits_list, names = [], [], []
    t0 = time.perf_counter()
    for i, seed in enumerate(seeds):
        z = _sample_latent(pipe, seed, IMAGE_SIZE, IMAGE_SIZE, DTYPE)
        bits = bits_from_latent(z)
        with torch.no_grad():
            out = pipe(
                prompt=PROMPT,
                latents=z.to(device=DEVICE, dtype=DTYPE),
                guidance_scale=GUIDANCE_SCALE,
                num_inference_steps=NUM_INFERENCE_STEPS,
                height=IMAGE_SIZE,
                width=IMAGE_SIZE,
            )
        img = out.images[0].convert("RGB")
        images.append(img)
        bits_list.append(bits)
        names.append(f"sample_{i:03d}")
        torch.save(
            {"z": z.detach().cpu(), "seed": seed, "bits": torch.from_numpy(bits)},
            WATERMARKED_DIR / f"sample_{i:03d}_latent.pt",
        )
        print(f"generated {i+1}/{n} seed={seed}", flush=True)
    _release_pipeline(pipe)
    key = {
        "scheme": "gaussian_shading_sign",
        "note": "Bits are sign(z) of the generation latent. Not full ChaCha20 GS.",
        "model_id": MODEL_ID,
        "prompt": PROMPT,
        "seeds": seeds,
        "image_size": IMAGE_SIZE,
        "num_inference_steps": NUM_INFERENCE_STEPS,
        "guidance_scale": GUIDANCE_SCALE,
    }
    _save_corpus(images, bits_list, key, WATERMARKED_DIR)
    print(f"generated {n} watermarked images in {time.perf_counter()-t0:.1f}s -> {WATERMARKED_DIR}")
    return images, bits_list, names, key


images = bits_list = names = watermark_key = None

if EXTERNAL_IMAGE_DIR.is_dir() and EXTERNAL_BITS_PATH.is_file():
    images, bits_list, names, watermark_key = _load_corpus_from_dir(
        EXTERNAL_IMAGE_DIR, EXTERNAL_BITS_PATH, EXTERNAL_KEY_PATH
    )
    print(f"loaded EXTERNAL corpus: {len(images)} images from {EXTERNAL_IMAGE_DIR}")
elif (WATERMARKED_DIR / "watermark_bits.pt").is_file() and any(WATERMARKED_DIR.glob("sample_*.png")):
    images, bits_list, names, watermark_key = _load_corpus_from_dir(
        WATERMARKED_DIR,
        WATERMARKED_DIR / "watermark_bits.pt",
        WATERMARKED_DIR / "watermark_key.json",
    )
    print(
        f"loaded cached corpus: {len(images)} images from {WATERMARKED_DIR} "
        f"scheme={watermark_key.get('scheme')}"
    )
    if watermark_key.get("scheme") == "synthetic_placeholder" and DEVICE == "cuda":
        print("cached corpus is synthetic_placeholder but CUDA is available; regenerating.")
        images = bits_list = names = watermark_key = None

if images is None:
    rng_seeds = np.random.default_rng(SEED)
    seeds = [int(x) for x in rng_seeds.integers(0, 2**31 - 1, size=NUM_IMAGES)]
    images, bits_list, names, watermark_key = generate_watermarked_corpus(NUM_IMAGES, seeds)

if len(images) < 1:
    raise RuntimeError("No watermarked images available.")

if len(images) > NUM_IMAGES:
    images = images[:NUM_IMAGES]
    bits_list = bits_list[:NUM_IMAGES]
    names = names[:NUM_IMAGES]
elif len(images) < NUM_IMAGES:
    print(f"warning: only {len(images)} images available; proceeding with that count.")

NUM_IMAGES = len(images)
NUM_WATERMARK_IMAGES = min(NUM_WATERMARK_IMAGES, NUM_IMAGES)
print(
    f"corpus ready: n={NUM_IMAGES}, bit_shape={bits_list[0].shape}, "
    f"key_scheme={watermark_key.get('scheme')}"
)


## 4. Load frozen SD1.5 VAE


In [ ]:
from diffusers import AutoencoderKL

print(f"Loading VAE from {MODEL_ID} ...", flush=True)
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=DTYPE)
vae = vae.to(DEVICE)
vae.eval()
for p in vae.parameters():
    p.requires_grad_(False)
print("VAE ready (frozen, eval).")


## 5. Cache original VAE latents and run the overlay experiment

Primary metric: **relative latent L2**. Also records absolute L2, cosine distance, RMSE, pixel MSE,
PSNR, SSIM, and LPIPS when available. Writes `results/cat_overlay_gs/results.csv`.


In [ ]:
original_latents = []
for i, img in enumerate(images):
    z = encode_vae_mean(vae, img, DEVICE, DTYPE)
    original_latents.append(z)
    if (i + 1) % 10 == 0 or i == 0:
        print(f"encoded original {i+1}/{NUM_IMAGES} latent shape={tuple(z.shape)}")


def iter_overlay_specs(for_watermark: bool = False):
    sizes = WM_CAT_SIZES if for_watermark else CAT_SIZES
    opacities = WM_CAT_OPACITIES if for_watermark else CAT_OPACITIES
    positions = WM_CAT_POSITIONS if for_watermark else CAT_POSITIONS
    for size in sizes:
        for opacity in opacities:
            for position in positions:
                yield {
                    "family": "cat",
                    "condition": f"cat_s{size:.2f}_o{opacity:.2f}_{position}",
                    "size": float(size),
                    "opacity": float(opacity),
                    "position": position,
                    "control_param": float("nan"),
                }
    include_controls = (not for_watermark) or WM_INCLUDE_CONTROLS
    if include_controls:
        noise_list = NOISE_STDS if not for_watermark else NOISE_STDS[:1]
        shape_list = SHAPE_OPACITIES if not for_watermark else SHAPE_OPACITIES[:1]
        text_list = TEXT_OPACITIES if not for_watermark else TEXT_OPACITIES[:1]
        for std in noise_list:
            yield {
                "family": "noise",
                "condition": f"noise_std{std:.2f}",
                "size": float("nan"),
                "opacity": float("nan"),
                "position": "n/a",
                "control_param": float(std),
            }
        for op in shape_list:
            yield {
                "family": "shapes",
                "condition": f"shapes_o{op:.2f}",
                "size": float("nan"),
                "opacity": float(op),
                "position": "n/a",
                "control_param": float(op),
            }
        for op in text_list:
            yield {
                "family": "text",
                "condition": f"text_o{op:.2f}",
                "size": float("nan"),
                "opacity": float(op),
                "position": "n/a",
                "control_param": float(op),
            }


def apply_overlay(base: Image.Image, spec: dict, rng: np.random.Generator) -> Image.Image:
    if spec["family"] == "cat":
        return place_cat(base, cat_rgba, spec["size"], spec["opacity"], spec["position"], rng)
    if spec["family"] == "noise":
        return overlay_gaussian_noise(base, spec["control_param"], rng)
    if spec["family"] == "shapes":
        return overlay_random_shapes(base, spec["opacity"], rng)
    if spec["family"] == "text":
        return overlay_text(base, spec["opacity"], rng)
    raise ValueError(spec)


overlay_specs = list(iter_overlay_specs(for_watermark=False))
print(f"VAE overlay conditions: {len(overlay_specs)}")

rows = []
example_heatmaps = {}
t_start = time.perf_counter()
for img_idx, (img, z0, name) in enumerate(zip(images, original_latents, names)):
    for spec in overlay_specs:
        rng = np.random.default_rng(SEED + 1000 * img_idx + condition_salt(spec["condition"]))
        mod = apply_overlay(img, spec, rng)
        z1 = encode_vae_mean(vae, mod, DEVICE, DTYPE)
        lm = latent_metrics(z0, z1)
        im = image_metrics(img, mod, lpips_fn=lpips_fn, device=DEVICE)
        rows.append(
            {
                "image_id": img_idx,
                "image_name": name,
                **{k: spec[k] for k in ("family", "condition", "size", "opacity", "position", "control_param")},
                **lm,
                **im,
                "bit_accuracy_original": float("nan"),
                "bit_accuracy_modified": float("nan"),
                "bit_accuracy_delta": float("nan"),
            }
        )
        if (
            img_idx == 0
            and spec["family"] == "cat"
            and spec["position"] == "center"
            and abs(spec["size"] - 0.20) < 1e-9
            and spec["opacity"] in {0.05, 0.20, 0.80}
        ):
            example_heatmaps[spec["condition"]] = {
                "original": img.copy(),
                "modified": mod.copy(),
                "heatmap": latent_heatmap(z0, z1),
            }
            mod.save(OUTPUT_DIR / "examples" / f"{spec['condition']}.png")
    print(f"VAE done image {img_idx+1}/{NUM_IMAGES} ({len(overlay_specs)} conditions)", flush=True)

results = pd.DataFrame(rows)
results_path = OUTPUT_DIR / "results.csv"
results.to_csv(results_path, index=False)
print(f"wrote {results_path} with {len(results)} rows in {time.perf_counter()-t_start:.1f}s")
results.head()


## 6. Gaussian Shading bit accuracy (DDIM inversion subset)

Runs DiffEdit DDIM inversion on the first `NUM_WATERMARK_IMAGES` images for a **small** overlay
grid, then decodes with the sign rule (or `EXTERNAL_GS_DECODE_FN` if set).

This is real watermark extraction, not a VAE-displacement proxy. It is expensive.


In [ ]:
wm_specs = list(iter_overlay_specs(for_watermark=True))
print(f"watermark overlay conditions: {len(wm_specs)} (+ identity baseline per image)")


def invert_image(image: Image.Image, inverse_pipe) -> torch.Tensor:
    from experiments.generate_and_invert import _recovered_latent

    expected_shape = torch.Size([1, 4, IMAGE_SIZE // 8, IMAGE_SIZE // 8])
    with torch.no_grad():
        inverted = inverse_pipe.invert(
            prompt=PROMPT,
            image=image,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            inpaint_strength=INVERSION_STRENGTH,
            num_reg_steps=0,
        )
    z_hat = _recovered_latent(inverted.latents, expected_shape)
    return z_hat.detach().to(device="cpu", dtype=torch.float32).contiguous()


def decode_watermark_bits(image: Image.Image, z_hat: torch.Tensor, key: dict) -> np.ndarray:
    if EXTERNAL_GS_DECODE_FN is not None:
        return np.asarray(EXTERNAL_GS_DECODE_FN(image, key)).astype(np.uint8)
    return bits_from_latent(z_hat)


bit_rows = []

if DEVICE != "cuda" or watermark_key.get("scheme") == "synthetic_placeholder":
    reason = (
        "CUDA required for DDIM inversion"
        if DEVICE != "cuda"
        else "corpus is synthetic_placeholder, not real GS"
    )
    print(
        f"SKIP watermark extraction: {reason}. "
        "VAE results above are still valid. Bit-accuracy plots will be empty. "
        "Full ChaCha20 Gaussian Shading decode is not in this repo; "
        "set EXTERNAL_GS_DECODE_FN if you have one."
    )
    bit_df = pd.DataFrame(bit_rows)
else:
    from experiments.generate_and_invert import _load_inversion_pipeline, _release_pipeline

    print("Loading inversion pipeline...", flush=True)
    inverse = _load_inversion_pipeline(MODEL_ID, DTYPE, DEVICE)
    t0 = time.perf_counter()
    inv_dir = OUTPUT_DIR / "inversions"
    inv_dir.mkdir(parents=True, exist_ok=True)

    for img_idx in range(NUM_WATERMARK_IMAGES):
        img = images[img_idx]
        known = bits_list[img_idx]
        name = names[img_idx]

        z_hat0 = invert_image(img, inverse)
        torch.save(z_hat0, inv_dir / f"{name}_original.pt")
        bits0 = decode_watermark_bits(img, z_hat0, watermark_key)
        acc0 = bit_accuracy(bits0, known)
        bit_rows.append(
            {
                "image_id": img_idx,
                "image_name": name,
                "family": "baseline",
                "condition": "original",
                "size": float("nan"),
                "opacity": float("nan"),
                "position": "n/a",
                "control_param": float("nan"),
                "bit_accuracy_original": acc0,
                "bit_accuracy_modified": acc0,
                "bit_accuracy_delta": 0.0,
                "latent_l2_rel": 0.0,
            }
        )
        print(f"[{img_idx}] original bit_acc={acc0:.4f}", flush=True)

        for spec in wm_specs:
            rng = np.random.default_rng(SEED + 1000 * img_idx + condition_salt(spec["condition"]))
            mod = apply_overlay(img, spec, rng)
            z_hat = invert_image(mod, inverse)
            torch.save(z_hat, inv_dir / f"{name}_{spec['condition']}.pt")
            bits = decode_watermark_bits(mod, z_hat, watermark_key)
            acc = bit_accuracy(bits, known)
            match = results[(results.image_id == img_idx) & (results.condition == spec["condition"])]
            rel = float(match.iloc[0]["latent_l2_rel"]) if len(match) else float("nan")
            bit_rows.append(
                {
                    "image_id": img_idx,
                    "image_name": name,
                    "family": spec["family"],
                    "condition": spec["condition"],
                    "size": spec["size"],
                    "opacity": spec["opacity"],
                    "position": spec["position"],
                    "control_param": spec["control_param"],
                    "bit_accuracy_original": acc0,
                    "bit_accuracy_modified": acc,
                    "bit_accuracy_delta": acc - acc0,
                    "latent_l2_rel": rel,
                }
            )
            print(
                f"[{img_idx}] {spec['condition']}: bit_acc={acc:.4f} (delta={acc-acc0:+.4f})",
                flush=True,
            )

    _release_pipeline(inverse)
    bit_df = pd.DataFrame(bit_rows)
    bit_path = OUTPUT_DIR / "bit_accuracy.csv"
    bit_df.to_csv(bit_path, index=False)
    print(f"wrote {bit_path} ({len(bit_df)} rows) in {time.perf_counter()-t0:.1f}s")

    if len(bit_df):
        merge_cols = [
            "image_id",
            "condition",
            "bit_accuracy_original",
            "bit_accuracy_modified",
            "bit_accuracy_delta",
        ]
        upd = bit_df[bit_df.condition != "original"][merge_cols]
        results = results.drop(
            columns=["bit_accuracy_original", "bit_accuracy_modified", "bit_accuracy_delta"]
        )
        results = results.merge(upd, on=["image_id", "condition"], how="left")
        results.to_csv(results_path, index=False)
        print(f"updated {results_path} with bit-accuracy columns where available")

bit_df.head() if len(bit_df) else bit_df


## 7. Summaries and visualizations


In [ ]:
FIG = OUTPUT_DIR / "figures"


def mean_std(df: pd.DataFrame, group_cols: list[str], value: str) -> pd.DataFrame:
    g = df.groupby(group_cols, dropna=False)[value]
    out = g.agg(["mean", "std", "count"]).reset_index()
    out["std"] = out["std"].fillna(0.0)
    return out


cat_center = results[(results.family == "cat") & (results.position == "center")].copy()
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for size, block in cat_center.groupby("size"):
    summ = mean_std(block, ["opacity"], "latent_l2_rel")
    ax.errorbar(summ.opacity, summ["mean"], yerr=summ["std"], marker="o", label=f"size={size:.0%}")
ax.set_xlabel("opacity")
ax.set_ylabel("relative latent L2")
ax.set_title("Latent displacement vs opacity (cat, center)")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(FIG / "latent_vs_opacity_by_size.png", bbox_inches="tight")
plt.show()

summ_size = mean_std(cat_center, ["size"], "latent_l2_rel")
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(summ_size["size"], summ_size["mean"], yerr=summ_size["std"], marker="o")
ax.set_xlabel("cat size (fraction of width)")
ax.set_ylabel("relative latent L2")
ax.set_title("Latent displacement vs cat size (center, mean over opacities)")
fig.tight_layout()
fig.savefig(FIG / "latent_vs_size.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4.5))
if results["lpips"].notna().any():
    for family, block in results.groupby("family"):
        ax.scatter(block["lpips"], block["latent_l2_rel"], s=12, alpha=0.5, label=family)
    ax.set_xlabel("LPIPS")
    ax.set_title("Latent displacement vs LPIPS")
else:
    for family, block in results.groupby("family"):
        ax.scatter(block["pixel_mse"], block["latent_l2_rel"], s=12, alpha=0.5, label=family)
    ax.set_xlabel("pixel MSE")
    ax.set_title("Latent displacement vs pixel MSE (LPIPS unavailable)")
ax.set_ylabel("relative latent L2")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "latent_vs_perceptual.png", bbox_inches="tight")
plt.show()

if example_heatmaps:
    conds = sorted(example_heatmaps.keys())
    fig, axes = plt.subplots(len(conds), 3, figsize=(9, 3.0 * len(conds)))
    if len(conds) == 1:
        axes = np.array([axes])
    for r, cond in enumerate(conds):
        ex = example_heatmaps[cond]
        axes[r, 0].imshow(ex["original"])
        axes[r, 0].set_title("original")
        axes[r, 1].imshow(ex["modified"])
        axes[r, 1].set_title(cond)
        im = axes[r, 2].imshow(ex["heatmap"], cmap="magma")
        axes[r, 2].set_title("latent |d| heatmap")
        fig.colorbar(im, ax=axes[r, 2], fraction=0.046)
        for c in range(3):
            axes[r, c].axis("off")
    fig.suptitle("Example overlays and latent displacement heatmaps (image 0)")
    fig.tight_layout()
    fig.savefig(FIG / "examples_and_heatmaps.png", bbox_inches="tight")
    plt.show()
else:
    print("no example heatmaps cached (need center cat size=0.20 at opacities 0.05/0.20/0.80)")

if len(bit_df):
    cat_bits = bit_df[bit_df.family == "cat"].copy()
    if len(cat_bits):
        fig, ax = plt.subplots(figsize=(7, 4.5))
        for size, block in cat_bits.groupby("size"):
            summ = mean_std(block, ["opacity"], "bit_accuracy_modified")
            ax.errorbar(summ.opacity, summ["mean"], yerr=summ["std"], marker="o", label=f"size={size:.0%}")
        base = bit_df[bit_df.condition == "original"]["bit_accuracy_modified"].mean()
        ax.axhline(base, color="k", ls="--", lw=1, label=f"original mean={base:.3f}")
        ax.set_xlabel("opacity")
        ax.set_ylabel("bit accuracy")
        ax.set_title("Watermark bit accuracy vs opacity")
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(FIG / "bit_acc_vs_opacity.png", bbox_inches="tight")
        plt.show()

    scored = bit_df[bit_df.condition != "original"].dropna(subset=["latent_l2_rel"])
    if len(scored):
        fig, ax = plt.subplots(figsize=(6.5, 4.5))
        for family, block in scored.groupby("family"):
            ax.scatter(
                block["latent_l2_rel"],
                block["bit_accuracy_modified"],
                s=28,
                alpha=0.7,
                label=family,
            )
        ax.set_xlabel("relative latent L2 (VAE)")
        ax.set_ylabel("bit accuracy after inversion")
        ax.set_title("Bit accuracy vs latent displacement")
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(FIG / "bit_acc_vs_latent.png", bbox_inches="tight")
        plt.show()
else:
    print("No bit_df rows; skipping bit-accuracy figures.")

summary_family = (
    results.groupby("family")
    .agg(
        n=("latent_l2_rel", "count"),
        latent_l2_rel_mean=("latent_l2_rel", "mean"),
        latent_l2_rel_std=("latent_l2_rel", "std"),
        pixel_mse_mean=("pixel_mse", "mean"),
        psnr_mean=("psnr", "mean"),
        ssim_mean=("ssim", "mean"),
        lpips_mean=("lpips", "mean"),
    )
    .reset_index()
)
summary_family.to_csv(OUTPUT_DIR / "summary_by_family.csv", index=False)
print("summary by family:")
print(summary_family.to_string(index=False))

if len(bit_df):
    bit_summary = (
        bit_df.groupby(["family", "condition"], dropna=False)
        .agg(
            n=("bit_accuracy_modified", "count"),
            bit_acc_mean=("bit_accuracy_modified", "mean"),
            bit_acc_std=("bit_accuracy_modified", "std"),
            delta_mean=("bit_accuracy_delta", "mean"),
            latent_l2_rel_mean=("latent_l2_rel", "mean"),
        )
        .reset_index()
        .sort_values(["family", "bit_acc_mean"])
    )
    bit_summary.to_csv(OUTPUT_DIR / "summary_bit_accuracy.csv", index=False)
    print("bit accuracy summary:")
    print(bit_summary.to_string(index=False))


## 8. Final analysis


In [ ]:
lines = []
lines.append("## Findings")

top = results.sort_values("latent_l2_rel", ascending=False).head(8)
lines.append("### Largest latent changes (top rows)")
for _, r in top.iterrows():
    extra = f", LPIPS={r.lpips:.4f}" if np.isfinite(r.lpips) else ""
    lines.append(
        f"- {r.condition} on {r.image_name}: relL2={r.latent_l2_rel:.4f}, "
        f"MSE={r.pixel_mse:.4f}, PSNR={r.psnr:.1f}{extra}"
    )

fam = summary_family.set_index("family")
lines.append("### Family means (relative latent L2)")
for f, row in fam.iterrows():
    lines.append(f"- {f}: {row.latent_l2_rel_mean:.4f} +/- {row.latent_l2_rel_std:.4f}")

lines.append("### Cats vs random shapes at comparable perceptual distortion")
cat = results[results.family == "cat"]
shp = results[results.family == "shapes"]
if len(cat) and len(shp):
    ref = float(np.nanmedian(np.concatenate([cat["pixel_mse"].to_numpy(), shp["pixel_mse"].to_numpy()])))
    band = max(1e-6, 0.5 * float(results["pixel_mse"].std()))
    cat_n = cat[np.abs(cat.pixel_mse - ref) <= band]
    shp_n = shp[np.abs(shp.pixel_mse - ref) <= band]
    if len(cat_n) and len(shp_n):
        lines.append(
            f"- Near pixel_mse~={ref:.4f}: cat relL2={cat_n.latent_l2_rel.mean():.4f}, "
            f"shapes relL2={shp_n.latent_l2_rel.mean():.4f}"
        )
        if np.isfinite(cat_n.lpips).any() and np.isfinite(shp_n.lpips).any():
            lines.append(
                f"- Same band LPIPS: cat={cat_n.lpips.mean():.4f}, shapes={shp_n.lpips.mean():.4f}"
            )
    else:
        lines.append("- Not enough overlap in pixel_mse to compare cats vs shapes fairly.")
else:
    lines.append("- Missing cat or shapes rows.")

lines.append("### Watermark bit accuracy")
if len(bit_df) == 0:
    lines.append(
        "- No bit-accuracy results. DDIM inversion did not run, or the corpus is synthetic. "
        "Do not treat VAE displacement as BER."
    )
    lines.append(
        "- Missing integration: full cryptographic Gaussian Shading decode (ChaCha20 / message recovery) "
        "is not in this repository. Native decode used here is sign(z_hat) vs known sign(z)."
    )
else:
    base = bit_df[bit_df.condition == "original"]
    lines.append(
        f"- Original mean bit accuracy: {base.bit_accuracy_modified.mean():.4f} "
        f"+/- {base.bit_accuracy_modified.std():.4f}"
    )
    hurt = (
        bit_df[bit_df.condition != "original"]
        .groupby("condition")["bit_accuracy_delta"]
        .mean()
        .sort_values()
    )
    lines.append("- Conditions that reduce accuracy the most (mean delta):")
    for cond, delta in hurt.head(8).items():
        lines.append(f"  - {cond}: {delta:+.4f}")
    scored = bit_df[bit_df.condition != "original"].dropna(
        subset=["latent_l2_rel", "bit_accuracy_modified"]
    )
    if len(scored) >= 3:
        corr = float(np.corrcoef(scored.latent_l2_rel, scored.bit_accuracy_modified)[0, 1])
        lines.append(
            f"- Correlation(latent_l2_rel, bit_accuracy_modified) = {corr:.3f} "
            "(negative means larger VAE move accompanies lower bit accuracy)."
        )
    if watermark_key.get("scheme") == "gaussian_shading_sign":
        lines.append("- Decode scheme: repo-native sign-bit Gaussian Shading (not full paper crypto).")

meta = {
    "seed": SEED,
    "num_images_vae": NUM_IMAGES,
    "num_images_watermark": int(NUM_WATERMARK_IMAGES if len(bit_df) else 0),
    "model_id": MODEL_ID,
    "cat_png": str(CAT_PNG_PATH),
    "quick_mode": QUICK_MODE,
    "watermark_scheme": watermark_key.get("scheme"),
    "external_gs_decode": EXTERNAL_GS_DECODE_FN is not None,
    "lpips": lpips_fn is not None,
    "results_csv": str(results_path),
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

text = "\n".join(lines)
print(text)
(OUTPUT_DIR / "analysis_summary.md").write_text(text + "\n", encoding="utf-8")
